# 09. Hobby Leakage Investigation (Forensic Analysis)

## Purpose
We suspect the `insured_hobbies` column contains "Data Leakage" (information that gives away the answer unfairly).
In synthetic datasets, features like "Chess" or "Cross-fit" sometimes accidentally correlate 100% with Fraud due to how the data was generated.
We need to prove this hypothesis to safely remove the column.

## The Evidence We Need:
1.  **Reliance**: Does the model perform *worse* if we hide this column?
2.  **Direct Link**: Do specific hobbies (e.g., "Chess") have a suspicious fraud rate (e.g., 90% or 10%)?
3.  **Correlations**: Is "Chess" a proxy for "Incident Severity"?


In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import fbeta_score, recall_score, mutual_info_score, precision_score, make_scorer
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, '../src')
from utils import load_insurance_data

In [ ]:
# 1. Load Data
# We load the 'tree' preprocessed data
df = pd.read_csv('../data/processed/preprocesed_for_trees.csv')
print(f"Dataset Shape: {df.shape}")

# Verify insured_hobbies exists
if 'insured_hobbies' in df.columns:
    print("Found 'insured_hobbies' column.")
    print(f"Unique Hobbies: {df['insured_hobbies'].nunique()}")
else:
    raise ValueError("'insured_hobbies' column NOT found in dataset!")

### Step 1: Force-Inducing the Suspect
**What:** We need to update our data loading pipeline.
**Why:** Our standard pipeline *already* drops `insured_hobbies` because we suspected it earlier. To test if it is bad, we must first put it back in!


In [ ]:
# 2. Custom Preprocessing (To INCLUDE Hobbies)
# The default utils.get_column_groups DROPS hobbies. We need to define one that keeps it.

def get_column_groups_with_hobbies(X_train):
    num_auto = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_auto = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Binary flags correction (same as utils.py)
    bool_like = [
        'collision_type_missing', 'property_damage_missing',
        'police_report_available_missing', 'authorities_contacted_missing',
        'incident_weekend', 'is_holiday', 'holiday_window_2d'
    ]
    for c in bool_like:
        if c in cat_auto: cat_auto.remove(c)
        if c in X_train.columns and c not in num_auto: num_auto.append(c)
            
    # Ordered categoricals
    ord_cols = ['vehicle_age_bucket', 'vehicle_tier']
    ord_categories = [['new', 'mid', 'old'], ['low', 'mid', 'premium']]
    ord_cols = [c for c in ord_cols if c in X_train.columns]
    
    # OHE - Ensure insured_hobbies is included
    # utils.py explicitly removes it. We just accept everything remaining in cat_auto
    ohe_cols = [c for c in cat_auto if c not in ord_cols and c in X_train.columns]
    num_cols = [c for c in num_auto if c in X_train.columns]
    
    return {
        'num_cols': num_cols,
        'ohe_cols': ohe_cols,
        'ord_cols': ord_cols,
        'ord_categories': ord_categories
    }

# Setup Data Splits
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Configure Preprocessor
groups = get_column_groups_with_hobbies(X_train)
print(f"OHE Columns including hobbies: {groups['ohe_cols']}")

transformers = [
    ("num", RobustScaler(), groups['num_cols']),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), groups['ohe_cols'])
]
if groups['ord_cols']:
    transformers.append((
        "ord", 
        OrdinalEncoder(categories=groups['ord_categories'], handle_unknown='use_encoded_value', unknown_value=-1), 
        groups['ord_cols']
    ))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop', verbose_feature_names_out=True)

### Step 2: Establish a Baseline
**What:** Train a Random Forest model *including* the `insured_hobbies`.
**Goal:** specific F2 score (e.g., 0.85). If we remove hobbies later and the score drops to 0.60, we know the model was cheating using the hobbies.


In [ ]:
# 3. Train Baseline Model
rf = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'))
])

print("Training RandomForest with Hobbies...")
rf.fit(X_train, y_train)

preds = rf.predict(X_test)
f2 = fbeta_score(y_test, preds, beta=2)
print(f"Baseline Test F2 (with Hobbies): {f2:.4f}")

### Step 3: The "Top Features" List
**What:** Ask the random forest what it thinks is important.
**Interpretation:** If `processed__ohe__insured_hobbies_chess` appears in the Top 5, it is a huge red flag. A hobby should count for very little compared to "Accident Severity".


In [ ]:
# 4. Feature Importance (MDI)
# Check if hobbies appear in the top features

feature_names = preprocessor.get_feature_names_out()
importances = rf.named_steps['clf'].feature_importances_
feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(25)

plt.figure(figsize=(10, 8))
sns.barplot(data=feat_imp, y='feature', x='importance', palette='viridis')
plt.title("Top 25 Features (MDI)")
plt.tight_layout()
plt.show()

# Highlight Hobby features
hobby_feats = [f for f in feature_names if 'processed__ohe__insured_hobbies' in f or 'insured_hobbies' in f]
print(f"Total Hobby OHE features: {len(hobby_feats)}")
print("Top Ranked Hobby Features:")
print(feat_imp[feat_imp['feature'].isin(hobby_feats)])

### Step 4: The Shuffle Test (Permutation Importance)
**What:** We take the `insured_hobbies` column and randomly shuffle the rows.
**Why:** This breaks any real relationship. If the model performance crashes (high importance), it proves the model relied heavily on that specific column.


In [ ]:
# 5. Permutation Importance (The 'Shuffle Test')
# This shuffles the ORIGINAL column 'insured_hobbies' (pre-OHE) and measures drop in F2 score.
# This tells us the TOTAL importance of the 'insured_hobbies' variable.

scorer = make_scorer(fbeta_score, beta=2)

print("Running Permutation Importance (this may take a minute)...")
perm_result = permutation_importance(
    rf, X_test, y_test, 
    scoring=scorer, 
    n_repeats=10, 
    random_state=42, 
    n_jobs=-1
)

perm_imp_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=perm_imp_df.head(20), y='feature', x='importance_mean', palette='coolwarm')
plt.title("Permutation Importance (Drop in F2 Score)")
plt.tight_layout()
plt.show()

print("Feature Importance of 'insured_hobbies':")
print(perm_imp_df[perm_imp_df['feature'] == 'insured_hobbies'])

# Interpretation:
# If importance_mean is HIGH positive -> The model RELIES on hobbies.
# If importance_mean is approx 0 -> The model ignores hobbies.

### Step 5: Direct Dependency Check
**What:** Calculate "Mutual Information" and specific Fraud Rates per hobby.
**Interpretation:**
*   **Global Fraud Rate**: ~25%
*   **Suspicious**: If "Camping" has a 90% fraud rate, that's almost certainly a data generation error (Leakage). No hobby makes you a criminal!


In [ ]:
# 6. Mutual Information Analysis
# Does any specific hobby strongly predict Fraud?

results = []
hobbies = df['insured_hobbies'].unique()

for hobby in hobbies:
    # Create binary indicator
    is_hobby = (df['insured_hobbies'] == hobby).astype(int)
    
    # Calculate MI with target
    mi = mutual_info_score(is_hobby, df['target'])
    
    # Calculate Fraud Rate for this hobby
    fraud_rate = df[df['insured_hobbies'] == hobby]['target'].mean()
    count = df[df['insured_hobbies'] == hobby].shape[0]
    
    results.append({
        'hobby': hobby,
        'mi_score': mi,
        'fraud_rate': fraud_rate,
        'count': count
    })

mi_df = pd.DataFrame(results).sort_values('mi_score', ascending=False)
global_fraud_rate = df['target'].mean()

print(f"Global Fraud Rate: {global_fraud_rate:.2%}")
print("Top Hobbies by Mutual Information:")
print(mi_df.head(10))

# Visualization
plt.figure(figsize=(12, 6))
sns.barplot(data=mi_df.head(20), x='mi_score', y='hobby', palette='magma')
plt.title("Hobby Mutual Information with Fraud Target")
plt.show()

### Step 6: The Accomplice Check
**What:** We check if the Top Suspect Hobby correlates with other features.
**Why:** Maybe "Chess" is only entered when "Incident Severity" is "Major". This helps us understand *how* the leakage happened.


In [ ]:
# 7. Correlation Analysis (Leakage Check)
# Are hobbies suspiciously correlated with other features?
# We'll look at the top hobby (highest MI) and see what else it correlates with.

top_hobby = mi_df.iloc[0]['hobby']
print(f"Analyzing correlations for top suspect hobby: '{top_hobby}'")

df['is_top_hobby'] = (df['insured_hobbies'] == top_hobby).astype(int)

# Calculate correlation with all numeric columns
corrs = df.select_dtypes(include=[np.number]).corrwith(df['is_top_hobby']).sort_values(ascending=False)

print("Top Positive Correlations:")
print(corrs.head(10))
print("\nTop Negative Correlations:")
print(corrs.tail(10))

# Plotly Visualization
import plotly.express as px

# Prepare data for plotting
# Limit to top/bottom 10 for clarity, removing 'is_top_hobby' itself (corr=1.0)
top_pos = corrs.drop('is_top_hobby', errors='ignore').head(10)
top_neg = corrs.tail(10)
plot_df = pd.concat([top_pos, top_neg]).reset_index()
plot_df.columns = ['Feature', 'Correlation']

# Plot
fig = px.bar(
    plot_df, 
    x='Correlation', 
    y='Feature', 
    orientation='h',
    color='Correlation',
    # Red for positive correlation, Blue for negative
    color_continuous_scale='RdBu_r', 
    title=f"Top Correlations with '{top_hobby}'"
)

# Sort by correlation value for cleaner view
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()


# Analysis Summary
1. **Model Reliance**: If Permutation Importance for `insured_hobbies` is > 0, the model is using it.
2. **Specific Leakage**: If specific hobbies have high MI scores, they are "giveaways" for fraud.
3. **Conclusion**: Use these results to justify removing `insured_hobbies` as synthetic noise/leakage.